# Predict which customers will leave(churn)

[Data Source](https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv#L7044)

In [67]:
import tensorflow as tf 
import numpy as np 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
import pandas as pd

### **Loading data**

In [68]:
df=pd.read_csv('data\Telco-Customer-Churn.csv')

In [69]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [70]:
rows,columns =df.shape
print(f"{rows} and {columns} columns")

7043 and 21 columns


In [71]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [72]:
df.isnull().sum()/len(df)*100

customerID          0.0
gender              0.0
SeniorCitizen       0.0
Partner             0.0
Dependents          0.0
tenure              0.0
PhoneService        0.0
MultipleLines       0.0
InternetService     0.0
OnlineSecurity      0.0
OnlineBackup        0.0
DeviceProtection    0.0
TechSupport         0.0
StreamingTV         0.0
StreamingMovies     0.0
Contract            0.0
PaperlessBilling    0.0
PaymentMethod       0.0
MonthlyCharges      0.0
TotalCharges        0.0
Churn               0.0
dtype: float64

In [73]:
values = df["TotalCharges"].unique() 
for index, value in enumerate(values):
    if value == ' ':
        print(index)
        break 
    print(f"{index}: VALUE: {value}: converted values => {float(value)}")

0: VALUE: 29.85: converted values => 29.85
1: VALUE: 1889.5: converted values => 1889.5
2: VALUE: 108.15: converted values => 108.15
3: VALUE: 1840.75: converted values => 1840.75
4: VALUE: 151.65: converted values => 151.65
5: VALUE: 820.5: converted values => 820.5
6: VALUE: 1949.4: converted values => 1949.4
7: VALUE: 301.9: converted values => 301.9
8: VALUE: 3046.05: converted values => 3046.05
9: VALUE: 3487.95: converted values => 3487.95
10: VALUE: 587.45: converted values => 587.45
11: VALUE: 326.8: converted values => 326.8
12: VALUE: 5681.1: converted values => 5681.1
13: VALUE: 5036.3: converted values => 5036.3
14: VALUE: 2686.05: converted values => 2686.05
15: VALUE: 7895.15: converted values => 7895.15
16: VALUE: 1022.95: converted values => 1022.95
17: VALUE: 7382.25: converted values => 7382.25
18: VALUE: 528.35: converted values => 528.35
19: VALUE: 1862.9: converted values => 1862.9
20: VALUE: 39.65: converted values => 39.65
21: VALUE: 202.25: converted values => 2

#### **Descent cleaning**

In [74]:
df_2 = df

In [75]:
df_2["TotalCharges"] = pd.to_numeric(df_2["TotalCharges"], errors="coerce") # coerce basically enable the alg. to be able to replace what cannot be changed into numerical with NaN (not a number)

In [76]:
df_2.iloc[470: 500][["customerID","TotalCharges"]]

,customerID,TotalCharges
470,4933-IKULF,330.60
471,3583-EKAPL,55.00
472,1304-BCCFO,564.40
473,4104-PVRPS,1315.35
474,9399-APLBT,74.70
475,2359-KMGLI,1861.50
476,3780-DDGSE,2747.20
477,4431-EDMIQ,554.05
478,0306-JAELE,453.40
479,6227-HWPWX,994.80


In [77]:
df_2.iloc[488]

customerID                         4472-LVYGI
gender                                 Female
SeniorCitizen                               0
Partner                                   Yes
Dependents                                Yes
tenure                                      0
PhoneService                               No
MultipleLines                No phone service
InternetService                           DSL
OnlineSecurity                            Yes
OnlineBackup                               No
DeviceProtection                          Yes
TechSupport                               Yes
StreamingTV                               Yes
StreamingMovies                            No
Contract                             Two year
PaperlessBilling                          Yes
PaymentMethod       Bank transfer (automatic)
MonthlyCharges                          52.55
TotalCharges                              NaN
Churn                                      No
Name: 488, dtype: object

In [78]:
df_2 = df_2.drop(columns =["customerID"]) # we drop the customer id

In [79]:
df_2.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [80]:
df.isna().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [81]:
df_2 = df_2.dropna() # we drop the rows with NaN values

In [82]:
df_2.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [83]:
df_2=pd.get_dummies(df_2,drop_first=True) # we convert the categorical variables into numerical variables

In [84]:
df_2.dtypes

SeniorCitizen                              int64
tenure                                     int64
MonthlyCharges                           float64
TotalCharges                             float64
gender_Male                                 bool
Partner_Yes                                 bool
Dependents_Yes                              bool
PhoneService_Yes                            bool
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No internet service             bool
TechSupport_Yes                             bool
StreamingTV_No inter

In [85]:
df_2["Churn_Yes"].value_counts()

Churn_Yes
False    5163
True     1869
Name: count, dtype: int64

In [86]:
df_2["Churn"]= (df_2["Churn_Yes"] == 1).astype(int) # we create a new column called churn which is equal to 1 if the customer has churned and 0 if the customer has not churned

In [87]:
df_2["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [88]:
X =df_2.drop(columns="Churn").values.astype("float32") # we drop the churn column from the dataset and store it in X
y = df_2["Churn"].values
X, y

(array([[  0.  ,   1.  ,  29.85, ...,   1.  ,   0.  ,   0.  ],
        [  0.  ,  34.  ,  56.95, ...,   0.  ,   1.  ,   0.  ],
        [  0.  ,   2.  ,  53.85, ...,   0.  ,   1.  ,   1.  ],
        ...,
        [  0.  ,  11.  ,  29.6 , ...,   1.  ,   0.  ,   0.  ],
        [  1.  ,   4.  ,  74.4 , ...,   0.  ,   1.  ,   1.  ],
        [  0.  ,  66.  , 105.65, ...,   0.  ,   0.  ,   0.  ]],
       shape=(7032, 31), dtype=float32),
 array([0, 0, 1, ..., 0, 1, 0], shape=(7032,)))

#### **Split then scale the data**

In [89]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

In [90]:
scaler = StandardScaler() # we create an instance of the StandardScaler class
scaler.fit(X_train) # we fit the scaler to the training data
X_train,X_test = scaler.transform(X_train), scaler.transform(X_test)# we transform the training data

In [91]:
X_train

array([[-0.4405457 , -0.833469  ,  0.44474947, ...,  1.3986266 ,
        -0.5367909 ,  1.6622266 ],
       [-0.4405457 , -0.508058  , -1.4921349 , ..., -0.71498716,
         1.8629229 , -0.60160273],
       [-0.4405457 , -1.2402327 , -0.12045064, ..., -0.71498716,
        -0.5367909 , -0.60160273],
       ...,
       [-0.4405457 ,  0.14276391,  0.92731786, ..., -0.71498716,
        -0.5367909 , -0.60160273],
       [-0.4405457 , -0.91482174,  0.03489676, ..., -0.71498716,
         1.8629229 , -0.60160273],
       [-0.4405457 , -1.2809091 , -1.3235666 , ..., -0.71498716,
        -0.5367909 ,  1.6622266 ]], shape=(4922, 31), dtype=float32)

## **Modelling**

In [92]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Input((X_train.shape[1],)), #  data is already flat hence we do not need to flatten the features trsnalte into an input

  # This is your first hidden layer n 128 neurons
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinhs

  # This is your first hidden layer n 64 neurons
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinh
  
  tf.keras.layers.Dense(10, activation='softmax')
])

In [93]:

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])


In [94]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [95]:
model.fit(X_train, y_train, epochs=30)

Epoch 1/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5297 - loss: 1.4900
Epoch 2/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8570 - loss: 0.4474
Epoch 3/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9482 - loss: 0.1649
Epoch 4/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9762 - loss: 0.0811
Epoch 5/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9898 - loss: 0.0426
Epoch 6/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9925 - loss: 0.0298
Epoch 7/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9943 - loss: 0.0207
Epoch 8/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9959 - loss: 0.0142
Epoch 9/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9972 - loss: 0.0115
Epoch 10/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9976 - loss: 0.0100
Epoch 11/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9986 - loss: 0.0070
Epoch 12/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

In [96]:
model_two = tf.keras.models.Sequential([
  tf.keras.layers.Input((X_train.shape[1],)), #  data is already flat hence we do not need to flatten the features trsnalte into an input

  # This is your first hidden layer n 32 neurons
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinhs

  # This is your first hidden layer n 16 neurons
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinh
  
  tf.keras.layers.Dense(1, activation='sigmoid')
])

In [97]:

model_two.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [ ]:

model_two.fit(X_train, y_train, epochs=10\
    
    
    
    
    \, batch_size=32)

Epoch 1/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 1.8142e-04
Epoch 2/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 1.5187e-04
Epoch 3/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 4.9484e-04
Epoch 4/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 2.5430e-04
Epoch 5/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 3.1835e-04
Epoch 6/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 1.1848e-04
Epoch 7/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 2.9592e-04
Epoch 8/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9998 - loss: 7.1336e-04
Epoch 9/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 6.7982e-05
Epoch 10/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 1.3012e-04


In [99]:
probs = model_two.predict(X_test).ravel()

66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
